[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/openai_tracing_tutorial.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪与评估结构化数据提取服务</h1>

本教程将：

- 使用 OpenAI 的[工具调用](https://platform.openai.com/docs/assistants/tools/function-calling)执行结构化数据提取，将非结构化输入（例如自然语言用户请求）转换为结构化格式（例如表格）
- 为 OpenAI 客户端添加埋点，以 [OpenInference 追踪](https://github.com/Arize-ai/open-inference-spec/blob/main/trace/spec/traces.md)格式记录数据
- 检查应用的追踪和 span，可视化追踪数据
- 导出追踪，评估结构化提取结果的质量

## 背景

OpenAI API 的强大功能之一是工具调用（函数调用）。用户通过 JSON Schema 和自然语言描述一个或多个函数的签名和参数，LLM 根据对话上下文决定何时调用各个函数，并提供参数值。除了将函数输入输出接入聊天消息序列，函数调用还适用于结构化数据提取：可以定义一个“函数”，描述期望的结构化输出格式。结构化数据提取可用于 ETL，或作为推荐系统等其他机器学习模型的输入。

虽然精心编写提示词也可能生成结构化输出，但函数调用更可靠，更容易生成符合特定格式的输出。详情请参阅 [OpenAI 文档](https://platform.openai.com/docs)。

开始吧！

ℹ️ 本 Notebook 需要 OpenAI API 密钥。

## 1. 安装依赖并导入库

安装依赖。

In [ ]:
!pip install "openai>=1.0.0" arize-phoenix arize-phoenix-client jsonschema openinference-instrumentation-openai

导入库。

In [ ]:
import json
import os
from getpass import getpass
from typing import Any, Dict, Literal, TypedDict

import jsonschema
import pandas as pd
from openai import OpenAI

pd.set_option("display.max_colwidth", None)

## 2. 配置 OpenAI API 密钥并实例化 OpenAI 客户端

如果尚未通过环境变量设置 OpenAI API 密钥，请在此设置。

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")
client = OpenAI(api_key=openai_api_key)

## 3. 为 OpenAI 客户端添加埋点

使用以 OpenInference 格式发送遥测数据的追踪器，为 OpenAI 客户端添加埋点。[OpenInference](https://arize-ai.github.io/open-inference-spec/trace/) 是用于捕获和存储 LLM 应用追踪的开放标准，使应用能够无缝接入 Phoenix 等 LLM 可观测性解决方案。

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

tracer_provider = register()
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

## 4. 在后台运行 Phoenix

以后台会话启动 Phoenix，收集添加埋点的 OpenAI 客户端发出的追踪数据。

In [ ]:
import phoenix as px

(session := px.launch_app()).view()

## 5. 提取结构化数据

从下方十条旅行请求中提取结构化数据。

In [ ]:
travel_requests = [
    "Can you recommend a luxury hotel in Tokyo with a view of Mount Fuji for a romantic honeymoon?",
    "I'm looking for a mid-range hotel in London with easy access to public transportation for a solo backpacking trip. Any suggestions?",
    "I need a budget-friendly hotel in San Francisco close to the Golden Gate Bridge for a family vacation. What do you recommend?",
    "Can you help me find a boutique hotel in New York City with a rooftop bar for a cultural exploration trip?",
    "I'm planning a business trip to Tokyo and I need a hotel near the financial district. What options are available?",
    "I'm traveling to London for a solo vacation and I want to stay in a trendy neighborhood with great shopping and dining options. Any recommendations for hotels?",
    "I'm searching for a luxury beachfront resort in San Francisco for a relaxing family vacation. Can you suggest any options?",
    "I need a mid-range hotel in New York City with a fitness center and conference facilities for a business trip. Any suggestions?",
    "I'm looking for a budget-friendly hotel in Tokyo with easy access to public transportation for a backpacking trip. What do you recommend?",
    "I'm planning a honeymoon in London and I want a luxurious hotel with a spa and romantic atmosphere. Can you suggest some options?",
]

OpenAI API 使用 [JSON Schema](https://json-schema.org/) 和自然语言描述来指定所调用函数的签名。本例将描述一个函数，从非结构化文本输入中记录以下属性：

- **location：** 期望的目的地
- **budget_level：** 预算偏好类别
- **purpose：** 旅行目的

JSON Schema 可以定义输出中每个字段的类型，还能为分类输出枚举有效值。因此，OpenAI 函数调用可以用于以前可能需要命名实体识别（NER）或分类模型完成的任务。

In [ ]:
parameters_schema = {
    "type": "object",
    "properties": {
        "location": {
            "type": "string",
            "description": '期望目的地。尽可能使用城市、州和国家格式。若未提供目的地，返回 "unstated"。',
        },
        "budget_level": {
            "type": "string",
            "enum": ["low", "medium", "high", "not_stated"],
            "description": '期望预算档次。若未提供预算档次，返回 "not_stated"。',
        },
        "purpose": {
            "type": "string",
            "enum": ["business", "pleasure", "other", "non_stated"],
            "description": '旅行目的。若未提供目的，返回 "not_stated"。',
        },
    },
    "required": ["location", "budget_level", "purpose"],
}
tool_schema = {
    "type": "function",
    "function": {
        "name": "record_travel_request_attributes",
        "description": "记录旅行请求的属性",
        "parameters": parameters_schema,
    },
}
system_message = (
    "你是负责解析并记录用户旅行请求属性的助手。"
)


def extract_raw_travel_request_attributes_string(
    travel_request: str,
    tool_schema: Dict[str, Any],
    system_message: str,
    client: OpenAI,
    model: str = "gpt-4o",
) -> str:
    chat_completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": travel_request},
        ],
        tools=[tool_schema],
        # 默认情况下，LLM 根据对话上下文决定是否调用函数。
        # 下方代码强制 LLM 调用函数，使输出符合模式。
        tool_choice={"type": "function", "function": {"name": tool_schema["function"]["name"]}},
    )
    tool_call = chat_completion.choices[0].message.tool_calls[0]
    return tool_call.function.arguments

执行提取。

In [ ]:
raw_travel_attributes_column = []
for travel_request in travel_requests:
    print("旅行请求：")
    print("==============")
    print(travel_request)
    print()
    raw_travel_attributes = extract_raw_travel_request_attributes_string(
        travel_request, tool_schema, system_message, client
    )
    raw_travel_attributes_column.append(raw_travel_attributes)
    print("原始旅行属性：")
    print("=====================")
    print(raw_travel_attributes)
    print()
    print()

追踪数据应实时出现在 Phoenix 界面中。

In [ ]:
print(f"🔥🐦 若尚未打开，请打开 Phoenix 界面： {session.url}")

函数调用可以提升 LLM 输出的可靠性，但并不保证输出符合输入的 JSON schema，甚至无法保证输出能够被解析为 JSON。因此，建议先验证输出，再将其传给下游流程。

In [ ]:
class TravelAttributes(TypedDict):
    location: str
    budget_level: Literal["low", "medium", "high", "not_stated"]
    purpose: Literal["business", "pleasure", "other", "non_stated"]


def parse_travel_attributes(
    travel_attributes_string: str, schema: Dict[str, Any]
) -> TravelAttributes:
    """
    将原始旅行属性字符串解析为结构化旅行属性字典。

    Parameters:
    - travel_attributes_string (str): 原始旅行属性字符串。
    - schema (Dict[str, Any]): 用于验证的 JSON Schema。

    Returns:
    - TravelAttributes: 解析后的旅行属性。
    """
    travel_attributes = json.loads(travel_attributes_string)
    jsonschema.validate(instance=travel_attributes, schema=schema)
    return travel_attributes


records = []
for travel_request, raw_travel_attributes in zip(travel_requests, raw_travel_attributes_column):
    is_json_parseable = True
    conforms_to_json_schema = True
    travel_attributes = None
    try:
        travel_attributes = parse_travel_attributes(raw_travel_attributes, parameters_schema)
    except json.JSONDecodeError:
        is_json_parseable = False
        conforms_to_json_schema = False
    except jsonschema.ValidationError:
        conforms_to_json_schema = False
    records.append(
        {
            "travel_request": travel_request,
            "raw_travel_attributes": raw_travel_attributes,
            "is_json_parseable": is_json_parseable,
            "conforms_to_json_schema": conforms_to_json_schema,
            **(travel_attributes or {}),
        }
    )

output_df = pd.DataFrame(records)
output_df

# 6. 导出与评估追踪数据

Phoenix 收集的 OpenInference 追踪数据可以导出为 pandas dataframe，以便进一步分析和评估。

In [ ]:
from phoenix.client import Client

trace_df = Client().spans.get_spans_dataframe()
trace_df

## 7. 回顾

本教程中，你已经：

- 使用 OpenAI 函数调用构建了从非结构化文本提取结构化数据的服务
- 使用 OpenInference 追踪器为服务添加埋点
- 在 Phoenix 中检查遥测数据

后续将介绍使用 LLM 评估来衡量服务表现的技巧。